# 9. Penalties while defending pace


### Purpose

Check the opponent's possession-sequence pace preceding a penalty taken. The coaching question is: **does this team take penalties while defending faster puck movement?**

### Expected outcome

A penalty-level table, pooled slower/middle/faster pace bands, penalty-type and team summaries, and a descriptive comparison with observed opponent movement exposure. Keep own-possession penalties and unknown preceding pace visible in coverage. Associations identify defensive sequences to review, not evidence that pace caused an infraction.


## 9.1. Setup and load processed data

Reuse augmented events and transition pace. Pool sources, retain provenance, and focus on 5v5 as in notebooks 07-08. Penalty rows reflect recorded manpower, which may not perfectly identify the moment of the infraction.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 9.2. Calculate sequence pace preceding penalties

Use the inferred possession sequence containing the penalty, trimmed to transitions ending before its event ID. Do not substitute the preceding sequence if the current one has no history, and do not use full sequence pace that can include later movement.

Penalties do not establish ownership in notebook 02. A penalty is classified as defending when the sequence owner is the opponent of the penalized team. Own-possession penalties are retained separately. Sequence pace means distance divided by modeled movement duration, with no fixed lookback. Unknown/zero-duration pace remains missing.

Preserve every penalty record. For exposure comparisons, group same-team penalties at the same recorded game/period/clock into one incident; this prevents double-minor or multiple assessment rows from counting as separate pressure episodes. The grouping is a convention, not verified incident tagging.


In [2]:
penalties = events.loc[events["event"].eq("Penalty Taken") & events["is_5v5"]].copy()
penalties["penalty_type"] = penalties["detail_1"].fillna("Unknown")
penalties["team_name"] = penalties["team_id"].map(teams.set_index("team_id")["team_name"])
penalties["player_name"] = penalties["player_id"].map(players.set_index("player_id")["player_name"])
penalties["game_date"] = penalties["game_id"].map(games.set_index("game_id")["game_date"])
penalties["defending"] = penalties["possession_team_id"].eq(penalties["opponent_team_id"])
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()].copy()
movement_groups = {
    pid: group[["end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
pace_rows = []
for penalty in penalties.itertuples(index=False):
    movement = movement_groups.get(penalty.possession_id)
    if movement is not None:
        movement = movement[movement[:, 0] < penalty.event_id]
    count = 0 if movement is None else len(movement)
    distance = 0.0 if movement is None else float(movement[:, 1].sum())
    duration = 0.0 if movement is None else float(movement[:, 2].sum())
    pace_rows.append({
        "event_id": penalty.event_id, "preceding_transitions": count,
        "preceding_distance_ft": distance, "preceding_movement_seconds": duration,
        "preceding_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "pace_status": "eligible" if duration > 0 else "no_history" if count == 0 else "zero_duration",
    })
penalties = penalties.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
display(penalties.groupby(["defending", "pace_status"]).size().rename("penalty_records").reset_index())


,defending,pace_status,penalty_records
0,False,eligible,28
1,False,no_history,124
2,False,zero_duration,1
3,True,eligible,54
4,True,no_history,15


## 9.3. Define pace bands using defending exposure

Counts of penalties alone cannot show whether faster possession has a higher penalty rate. Build a baseline from eligible 5v5 opponent movement intervals, including intervals in sequences with no penalty. Each interval receives the sequence pace known at its start: pooled movement before that interval, excluding the current movement.

Define pooled tertiles from those available preceding-pace estimates, then apply the same boundaries to penalties. Extreme values fall in the outer bands. Empty-history intervals have no band and do not supply exposure.

Exposure is **modeled opponent movement time**, not total defending time or time on ice. The rate is incidents per 60 minutes of this observed exposure, not a calibrated probability or game-clock penalty rate. This comparison is descriptive; event density, unobserved loose-puck time, repeated teams, and competition differences can affect it.


In [3]:
sequence_context = events.groupby("possession_id").agg(
    is_5v5=("is_5v5", "first"), opponent_team_id=("opponent_team_id", "first"),
    first_event_team_id=("team_id", "first"), owner=("possession_team_id", "first"),
    game_id=("game_id", "first"),
)
# Determine the defending team from game participants, not the first credited event.
game_lookup = games.set_index("game_id")
home = sequence_context["game_id"].map(game_lookup["home_team_id"])
away = sequence_context["game_id"].map(game_lookup["away_team_id"])
sequence_context["defending_team_id"] = away.where(sequence_context["owner"].eq(home), home)
exposure = valid_movements.loc[
    valid_movements["possession_id"].isin(sequence_context.index[sequence_context["is_5v5"]])
].sort_values("end_event_id").copy()
for field in ["distance_ft", "modeled_elapsed_seconds"]:
    exposure[f"prior_{field}"] = exposure.groupby("possession_id")[field].cumsum() - exposure[field]
exposure["preceding_pace_ft_s"] = exposure["prior_distance_ft"] / exposure["prior_modeled_elapsed_seconds"].where(
    exposure["prior_modeled_elapsed_seconds"].gt(0)
)
exposure["team_id"] = exposure["possession_id"].map(sequence_context["defending_team_id"])
exposure = exposure.loc[exposure["preceding_pace_ft_s"].notna() & exposure["modeled_elapsed_seconds"].gt(0)].copy()
_, pace_edges = pd.qcut(exposure["preceding_pace_ft_s"], q=3, labels=False, retbins=True, duplicates="drop")
band_edges = pace_edges.copy()
band_edges[0], band_edges[-1] = -np.inf, np.inf
band_names = ["Slower", "Middle", "Faster"][:len(band_edges)-1]
exposure["pace_band"] = pd.cut(exposure["preceding_pace_ft_s"], bins=band_edges, labels=band_names)
penalties["pace_band"] = pd.cut(penalties["preceding_pace_ft_s"], bins=band_edges, labels=band_names)
pace_boundaries = pd.DataFrame({"pace_band": band_names, "lower_ft_s": band_edges[:-1], "upper_ft_s": band_edges[1:]})
display(pace_boundaries.round(2))

pressure_penalties = penalties.loc[penalties["defending"] & penalties["pace_status"].eq("eligible")].copy()
penalty_incidents = pressure_penalties.drop_duplicates(["game_id", "period", "clock_seconds", "team_id"]).copy()
print(f"5v5 records: {len(penalties)} | defending with pace: {len(pressure_penalties)} | incidents: {len(penalty_incidents)}")
print(f"Band-assigned observed movement exposure: {exposure['modeled_elapsed_seconds'].sum() / 60:.1f} minutes")


,pace_band,lower_ft_s,upper_ft_s
0,Slower,-inf,15.30
1,Middle,15.30,22.39
2,Faster,22.39,inf


5v5 records: 222 | defending with pace: 54 | incidents: 53
Band-assigned observed movement exposure: 746.0 minutes


## 9.4. Compare penalty incidents with exposure

Show incidents per 60 minutes of observed opponent movement in slower/middle/faster bands, without uncertainty intervals. Include exposure minutes, incident count, and contributing games. A zero-exposure group has an undefined rate, not zero.

The predictor is available before each movement interval and before each penalty. This is a simple descriptive comparison, not a survival model. Differences in sequence age and manpower tagging remain possible explanations.


In [4]:
rate_summary = exposure.groupby("pace_band", observed=True).agg(
    movement_seconds=("modeled_elapsed_seconds", "sum"),
    movement_intervals=("end_event_id", "size"), games=("game_id", "nunique"),
)
rate_summary["incidents"] = penalty_incidents.groupby("pace_band", observed=True).size().reindex(rate_summary.index, fill_value=0)
rate_summary["exposure_minutes"] = rate_summary["movement_seconds"] / 60
rate_summary["incidents_per_60_movement_minutes"] = (
    rate_summary["incidents"] * 3600 / rate_summary["movement_seconds"].where(rate_summary["movement_seconds"].gt(0))
)
rate_summary = rate_summary.reset_index()
display(rate_summary.round(2))
rate_fig = px.bar(
    rate_summary, x="pace_band", y="incidents_per_60_movement_minutes", color="pace_band",
    category_orders={"pace_band": band_names},
    color_discrete_map={"Slower":"#93c5fd", "Middle":"#3b82f6", "Faster":"#1d4ed8"},
    hover_data=["incidents", "exposure_minutes", "games"],
    labels={"pace_band":"Opponent preceding sequence pace", "incidents_per_60_movement_minutes":"Incidents / 60 observed movement minutes"},
    title="5v5 penalties while defending pace",
)
rate_fig.update_layout(template="plotly_white", height=450, showlegend=False)
rate_fig.show()


,pace_band,movement_seconds,movement_intervals,games,incidents,exposure_minutes,incidents_per_60_movement_minutes
0,Slower,12024.98,10003,34,13,200.42,3.89
1,Middle,14535.26,10003,34,20,242.25,4.95
2,Faster,18197.07,10003,34,20,303.28,3.96


## 9.5. Inspect penalty types and teams

Retain all recorded penalty types. Highlight hooking, holding, holding the stick, tripping, and interference as a review group related to obstruction; this grouping is a hockey hypothesis, not proof of pace-induced infractions. Roughing and administrative penalties may have different explanations.

Type charts show record counts, not exposure-adjusted risk. Team rate tables use incidents and their own exposure denominators. Sparse team cells are for inspection, not rankings. Same-time incidents can contain several types, so type-record counts need not sum to incident counts.


In [5]:
OBSTRUCTION_TYPES = ["Hooking", "Holding", "Holding the stick", "Tripping", "Interference"]
pressure_penalties["obstruction_group"] = pressure_penalties["penalty_type"].isin(OBSTRUCTION_TYPES)
type_summary = pressure_penalties.groupby(["penalty_type", "pace_band"], observed=True).size().rename("records").reset_index()
display(type_summary)
type_fig = px.bar(type_summary, x="penalty_type", y="records", color="pace_band", barmode="group",
    category_orders={"pace_band":band_names}, title="Recorded defending penalties by type and opponent pace")
type_fig.update_layout(template="plotly_white", height=500)
type_fig.show()

team_rates = exposure.groupby(["team_id", "pace_band"], observed=True).agg(
    movement_seconds=("modeled_elapsed_seconds", "sum"), games=("game_id", "nunique"),
)
team_rates["incidents"] = penalty_incidents.groupby(["team_id", "pace_band"], observed=True).size().reindex(team_rates.index, fill_value=0)
team_rates["incidents_per_60_movement_minutes"] = team_rates["incidents"] * 3600 / team_rates["movement_seconds"].where(team_rates["movement_seconds"].gt(0))
team_rates = team_rates.reset_index()
team_rates["team_name"] = team_rates["team_id"].map(teams.set_index("team_id")["team_name"])
display(team_rates.round(2))


,penalty_type,pace_band,records
0,Charging,Faster,1
1,Closing hand on puck,Faster,1
2,Cross-checking,Slower,2
3,Cross-checking,Middle,3
4,Cross-checking,Faster,1
5,Face-off violation,Faster,1
6,Goalkeeper interference,Faster,1
7,Holding,Middle,1
8,Holding,Faster,1
9,Hooking,Slower,2


,team_id,pace_band,movement_seconds,games,incidents,incidents_per_60_movement_minutes,team_name
0,1,Slower,977.92,7,3,11.04,Boston Pride
1,1,Middle,1183.0,7,2,6.09,Boston Pride
2,1,Faster,1374.83,7,1,2.62,Boston Pride
3,2,Slower,989.67,6,1,3.64,Buffalo Beauts
4,2,Middle,1382.0,6,1,2.6,Buffalo Beauts
5,2,Faster,1850.0,6,4,7.78,Buffalo Beauts
6,3,Slower,792.25,4,0,0.0,Connecticut Whale
7,3,Middle,867.83,4,2,8.3,Connecticut Whale
8,3,Faster,1015.82,4,1,3.54,Connecticut Whale
9,4,Slower,587.58,3,1,6.13,Metropolitan Riveters


## 9.6. Review the sequences behind penalties

Select a pace band and inspect the penalized player, type, sequence owner, movement history, and event order. Check inferred ownership and whether the penalty was recorded after the actual infraction. Own-possession penalties remain available in `penalties` but do not enter the defending-pace comparison.

Before making a coaching recommendation, review sequences, compare equivalent penalty types, assess consistency across games, and check timing assumptions. A rate difference is a review signal, not evidence that faster puck movement caused penalties.


In [6]:
REVIEW_PACE_BAND = "Faster"
review_penalties = pressure_penalties.loc[pressure_penalties["pace_band"].eq(REVIEW_PACE_BAND)]
display(review_penalties[[
    "game_id", "game_date", "team_name", "player_name", "period", "clock_seconds",
    "event_id", "possession_id", "possession_team_id", "penalty_type",
    "preceding_pace_ft_s", "preceding_movement_seconds", "preceding_transitions",
]].head(20))


,game_id,game_date,team_name,player_name,period,clock_seconds,event_id,possession_id,possession_team_id,penalty_type,preceding_pace_ft_s,preceding_movement_seconds,preceding_transitions
4,3,2021-01-23,Minnesota Whitecaps,Emma Stauber,2,930.0,929,194,1,Hooking,29.455661,4.0,4
16,2,2021-01-23,Toronto Six,Julie Allen,2,381.0,6388,1255,4,Charging,22.666565,1.5,2
19,4,2021-01-24,Buffalo Beauts,Iveta Klimasova,2,1028.0,8351,1663,1,Tripping,22.971803,6.0,4
51,7,2021-01-26,Minnesota Whitecaps,Maddie Rowe,3,990.0,19545,3772,4,Hooking,23.441606,17.0,12
56,10,2021-01-27,Connecticut Whale,Emma Vlasic,2,1038.0,21243,4116,1,Holding,26.985838,4.0,6
65,9,2021-01-27,Buffalo Beauts,Kayla Meneghin,2,541.0,24022,4622,6,Roughing,35.733942,4.0,4
76,11,2021-01-30,Boston Pride,Taylor Wenczkowski,3,971.0,27002,5198,2,Face-off violation,22.524228,2.0,3
84,12,2021-01-30,Toronto Six,Mikyla Grant-Mentis,3,996.0,29520,5672,5,Too many men on the ice,28.474697,1.0,3
85,12,2021-01-30,Toronto Six,Taylor Woods,3,614.0,29749,5715,5,Tripping,38.592110,8.0,6
103,15,2021-02-01,Buffalo Beauts,Taylor Wasylk,3,387.0,37311,7175,1,Interference,32.350331,2.5,4
